# 🎵 Psychoacoustic Visual Recognition
## Train Vision Transformers on Spiral Cochlear Visualizations

This notebook lets you train modern neural networks (ViT, AST) to classify musical instruments from spiral cochlear visualizations.

**What you'll do:**
1. Generate spiral visualizations from audio (or use synthetic data)
2. Train a Vision Transformer classifier
3. Evaluate and visualize attention maps
4. Download your trained model

**Requirements:** GPU runtime (free T4 on Colab)

---

## 🚀 Setup

In [ ]:
# Check GPU availability
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Install dependencies
!pip install -q timm librosa tqdm pillow

print("Dependencies installed!")

In [ ]:
# Option 1: Upload your code from the project
# Uncomment and run if you have the modern_classifier folder

# from google.colab import files
# uploaded = files.upload()  # Upload the ZIP of modern_classifier
# !unzip modern_classifier.zip

# Option 2: Create the essential files here (default)
print("Creating project files...")

In [ ]:
%%writefile spiral_visualizer.py
"""
Spiral Cochlear Visualization Generator
"""
import numpy as np
from PIL import Image, ImageDraw
import colorsys

class SpiralVisualizer:
    def __init__(self, image_size=224, num_frequency_bins=128):
        self.image_size = image_size
        self.num_frequency_bins = num_frequency_bins
        self._compute_spiral_coordinates()
    
    def _compute_spiral_coordinates(self):
        center = self.image_size // 2
        max_radius = center * 0.9
        phi_max = 3.0 * 2 * np.pi
        
        self.spiral_coords = []
        self.spiral_colors = []
        
        for i in range(self.num_frequency_bins):
            t = i / (self.num_frequency_bins - 1)
            phi = t * phi_max
            r = max_radius * np.sqrt(t)
            x = center + r * np.cos(phi)
            y = center + r * np.sin(phi)
            self.spiral_coords.append((x, y))
            
            hue = t * 0.75
            rgb = colorsys.hsv_to_rgb(hue, 1.0, 1.0)
            color = tuple(int(c * 255) for c in rgb)
            self.spiral_colors.append(color)
    
    def spectrum_to_image(self, spectrum):
        img = Image.new('RGB', (self.image_size, self.image_size), (0, 0, 0))
        draw = ImageDraw.Draw(img)
        max_radius = self.image_size / (self.num_frequency_bins * 0.3)
        
        for i, (coord, color) in enumerate(zip(self.spiral_coords, self.spiral_colors)):
            x, y = coord
            magnitude = spectrum[i]
            radius = magnitude * max_radius
            
            if radius > 0.5:
                h, s, v = colorsys.rgb_to_hsv(color[0]/255, color[1]/255, color[2]/255)
                v = max(0.3, magnitude)
                r, g, b = colorsys.hsv_to_rgb(h, s, v)
                adjusted_color = (int(r*255), int(g*255), int(b*255))
                draw.ellipse([x - radius, y - radius, x + radius, y + radius], fill=adjusted_color)
        
        return img


def generate_synthetic_data(output_dir, num_classes=5, samples_per_class=200):
    """Generate synthetic spiral images for training."""
    import os
    from pathlib import Path
    
    output_dir = Path(output_dir)
    class_names = ["piano", "guitar", "violin", "flute", "drums"][:num_classes]
    
    class_patterns = {
        "piano": {"freq_range": (0.2, 0.8), "brightness": 0.9, "spread": 0.3},
        "guitar": {"freq_range": (0.3, 0.7), "brightness": 0.8, "spread": 0.4},
        "violin": {"freq_range": (0.4, 0.9), "brightness": 0.85, "spread": 0.2},
        "flute": {"freq_range": (0.5, 0.95), "brightness": 0.75, "spread": 0.15},
        "drums": {"freq_range": (0.1, 0.5), "brightness": 0.95, "spread": 0.5},
    }
    
    visualizer = SpiralVisualizer()
    
    for split in ["train", "val"]:
        split_samples = samples_per_class if split == "train" else samples_per_class // 5
        
        for class_name in class_names:
            class_dir = output_dir / split / class_name
            class_dir.mkdir(parents=True, exist_ok=True)
            
            pattern = class_patterns[class_name]
            
            for i in range(split_samples):
                spectrum = np.zeros(visualizer.num_frequency_bins)
                freq_start = int(pattern["freq_range"][0] * visualizer.num_frequency_bins)
                freq_end = int(pattern["freq_range"][1] * visualizer.num_frequency_bins)
                
                num_peaks = np.random.randint(3, 8)
                for _ in range(num_peaks):
                    peak_pos = np.random.randint(freq_start, freq_end)
                    peak_width = int(pattern["spread"] * 20) + np.random.randint(5, 15)
                    peak_height = pattern["brightness"] * (0.7 + 0.3 * np.random.random())
                    
                    for j in range(max(0, peak_pos - peak_width), min(visualizer.num_frequency_bins, peak_pos + peak_width)):
                        distance = abs(j - peak_pos) / peak_width
                        spectrum[j] = max(spectrum[j], peak_height * np.exp(-distance**2 * 2))
                
                spectrum += np.random.random(visualizer.num_frequency_bins) * 0.1
                spectrum = np.clip(spectrum, 0, 1)
                
                img = visualizer.spectrum_to_image(spectrum)
                img.save(class_dir / f"sample_{i:04d}.jpg", quality=95)
            
            print(f"Generated {split_samples} samples for {split}/{class_name}")
    
    return class_names

In [ ]:
%%writefile vit_classifier.py
"""
Vision Transformer Classifier
"""
import torch
import torch.nn as nn
import timm

class ViTClassifier(nn.Module):
    def __init__(self, num_classes=5, model_name="vit_small_patch16_224", pretrained=True, dropout=0.1):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0)
        self.feature_dim = self.backbone.num_features
        self.classifier = nn.Sequential(
            nn.LayerNorm(self.feature_dim),
            nn.Dropout(dropout),
            nn.Linear(self.feature_dim, num_classes)
        )
        self.num_classes = num_classes
    
    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)


def create_model(num_classes=5, model_size="small", pretrained=True):
    model_names = {
        "tiny": "vit_tiny_patch16_224",
        "small": "vit_small_patch16_224",
        "base": "vit_base_patch16_224"
    }
    return ViTClassifier(num_classes, model_names[model_size], pretrained)

## 📊 Generate Training Data

In [ ]:
from spiral_visualizer import generate_synthetic_data

# Generate synthetic spiral visualizations
# Adjust num_classes and samples_per_class as needed
classes = generate_synthetic_data(
    "data", 
    num_classes=5, 
    samples_per_class=200
)

print(f"\nDataset created with classes: {classes}")

In [ ]:
# Visualize some samples
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

fig, axes = plt.subplots(1, 5, figsize=(15, 3))

for i, class_name in enumerate(classes):
    img_path = list(Path(f"data/train/{class_name}").glob("*.jpg"))[0]
    img = Image.open(img_path)
    axes[i].imshow(img)
    axes[i].set_title(class_name)
    axes[i].axis('off')

plt.suptitle("Sample Spiral Visualizations by Class")
plt.tight_layout()
plt.show()

## 🏋️ Training

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
from pathlib import Path
from tqdm.notebook import tqdm

# Dataset
class SpiralDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.data_dir = Path(data_dir)
        self.transform = transform
        self.samples = []
        self.classes = sorted([d.name for d in self.data_dir.iterdir() if d.is_dir()])
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        
        for class_name in self.classes:
            for img_path in (self.data_dir / class_name).glob("*.jpg"):
                self.samples.append((img_path, self.class_to_idx[class_name]))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

# Transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Create datasets
train_dataset = SpiralDataset("data/train", train_transform)
val_dataset = SpiralDataset("data/val", val_transform)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")
print(f"Classes: {train_dataset.classes}")

# Dataloaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

In [ ]:
from vit_classifier import create_model

# Create model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = len(train_dataset.classes)

model = create_model(
    num_classes=num_classes,
    model_size="small",  # Options: tiny, small, base
    pretrained=True
).to(device)

print(f"Model on: {device}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Training setup
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

# Training loop
num_epochs = 30
best_val_acc = 0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(num_epochs):
    # Training
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0
    
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        _, predicted = outputs.max(1)
        train_total += labels.size(0)
        train_correct += predicted.eq(labels).sum().item()
    
    # Validation
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
    
    scheduler.step()
    
    # Record history
    train_acc = 100. * train_correct / train_total
    val_acc = 100. * val_correct / val_total
    
    history['train_loss'].append(train_loss / len(train_loader))
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss / len(val_loader))
    history['val_acc'].append(val_acc)
    
    print(f"  Train: Loss={train_loss/len(train_loader):.4f}, Acc={train_acc:.2f}%")
    print(f"  Val: Loss={val_loss/len(val_loader):.4f}, Acc={val_acc:.2f}%")
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'classes': train_dataset.classes,
            'val_acc': val_acc
        }, 'best_model.pt')
        print(f"  ✓ Saved best model (acc: {val_acc:.2f}%)")

print(f"\nTraining complete! Best validation accuracy: {best_val_acc:.2f}%")

In [ ]:
# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'], label='Validation')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss')
ax1.legend()

ax2.plot(history['train_acc'], label='Train')
ax2.plot(history['val_acc'], label='Validation')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Accuracy')
ax2.legend()

plt.tight_layout()
plt.show()

## 📈 Evaluation

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Load best model
checkpoint = torch.load('best_model.pt')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Get predictions
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = outputs.max(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(10, 8))
sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=train_dataset.classes, yticklabels=train_dataset.classes)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=train_dataset.classes))

## 💾 Download Model

In [ ]:
from google.colab import files

# Download the trained model
files.download('best_model.pt')

print("Model downloaded! You can use it with:")
print("  python demo_realtime.py --checkpoint best_model.pt")

## 🔍 Attention Visualization (Optional)

In [ ]:
# Visualize what the model attends to
# This shows which parts of the spiral influence classification

def get_attention_map(model, image_tensor):
    """Extract attention from ViT."""
    model.eval()
    
    # Hook to capture attention
    attention = None
    def hook(module, input, output):
        nonlocal attention
        # output is (B, num_heads, N, N)
        attention = output.detach()
    
    # Register hook on last attention layer
    handle = model.backbone.blocks[-1].attn.attn_drop.register_forward_hook(hook)
    
    with torch.no_grad():
        _ = model(image_tensor)
    
    handle.remove()
    
    if attention is not None:
        # Average over heads, get CLS token attention to patches
        attn_weights = attention[0].mean(dim=0)  # (N, N)
        cls_attention = attn_weights[0, 1:]  # CLS to patches
        
        # Reshape to 2D
        num_patches = int(np.sqrt(cls_attention.shape[0]))
        attention_map = cls_attention.reshape(num_patches, num_patches)
        return attention_map.cpu().numpy()
    
    return None

# Visualize for a few samples
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

for i in range(5):
    image, label = val_dataset[i * 10]
    image_tensor = image.unsqueeze(0).to(device)
    
    # Get prediction
    with torch.no_grad():
        output = model(image_tensor)
        pred = output.argmax(1).item()
    
    # Get attention
    attn_map = get_attention_map(model, image_tensor)
    
    # Display original image
    img_np = image.permute(1, 2, 0).numpy()
    img_np = img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    img_np = np.clip(img_np, 0, 1)
    
    axes[0, i].imshow(img_np)
    axes[0, i].set_title(f"True: {train_dataset.classes[label]}\nPred: {train_dataset.classes[pred]}")
    axes[0, i].axis('off')
    
    # Display attention
    if attn_map is not None:
        from scipy.ndimage import zoom
        attn_map_resized = zoom(attn_map, 224 / attn_map.shape[0], order=1)
        axes[1, i].imshow(img_np)
        axes[1, i].imshow(attn_map_resized, cmap='hot', alpha=0.5)
        axes[1, i].set_title("Attention")
        axes[1, i].axis('off')

plt.suptitle("Model Attention Visualization")
plt.tight_layout()
plt.show()

---

## 🎉 Done!

You've trained a Vision Transformer on spiral cochlear visualizations.

**Next steps:**
1. Download your model and use it locally
2. Try with real audio data (use NSynth or IRMAS datasets)
3. Run the real-time demo: `python demo_realtime.py --checkpoint best_model.pt`

**To train on real audio:**
1. Upload your audio files to Colab
2. Use `librosa` to convert to spectrograms
3. Use `SpiralVisualizer` to generate spiral images
4. Train as above